## Tune the Alignment

# Parameter Sweeping for Optimal Alignment

Welcome to the fourth and final unit of our course! Throughout the previous units, we built a solid foundation: extracting features, using RANSAC to calculate a homography matrix while filtering out outliers, and warping images into a shared coordinate frame.

Up until now, our matching and RANSAC routines relied on fixed values (such as a Lowe's match ratio of `0.75` or a RANSAC threshold of `5.0`). These hardcoded "magic numbers" can cause an alignment pipeline to succeed on one scene and fail entirely on another.

In this lesson, we build a **parameter sweeper**—a diagnostic tool that systematically tests combinations of matching and geometric thresholds to determine optimal settings for any given image pair.

---

## Recall: The Feature Pipeline

Before running a parameter sweep, we load the images and compute their keypoints and descriptors using our helper functions:

```python
from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography

# Read the left and right images
left = read_color("left_image.jpg")
right = read_color("right_image.jpg")

# Detect keypoints and compute descriptors for both images
kp1, des1 = detect_and_compute(preprocess_for_features(left), method="sift")
kp2, des2 = detect_and_compute(preprocess_for_features(right), method="sift")

```

---

## The Two Key Parameters

Alignment quality depends primarily on two interacting thresholds:

* **Lowe's Ratio (`ratio`):** Controls the strictness of descriptor matching.
* A lower ratio (e.g., `0.65`) enforces stricter uniqueness, yielding fewer but higher-confidence matches.
* A higher ratio (e.g., `0.85`) yields more candidate matches at the cost of higher outlier rates.


* **RANSAC Threshold (`ransac_threshold`):** Defines the maximum reprojection error (in pixels) for a match to be classified as an inlier.
* Stricter values (e.g., `3.0` px) require tight geometric agreement.
* Looser values (e.g., `8.0` px) accommodate slight camera distortion, slight non-planarity, or lower keypoint localization accuracy.



The goal is to find a configuration that yields **sufficient total inliers alongside a high inlier ratio**, rather than merely maximizing raw match count.

---

## Building the Parameter Sweep

We evaluate parameter combinations using nested loops:

1. **Outer loop:** Iterates over Lowe's ratio thresholds and recomputes descriptor matches.
2. **Inner loop:** Evaluates multiple RANSAC reprojection thresholds on the current match set.

```python
# Print a table header to organize our output
print(f"{'ratio':>6} {'ransac':>8} {'matches':>8} {'inliers':>8} {'inlier_ratio':>13}")

# The outer loop tests different strictness levels for matching
for ratio in [0.65, 0.70, 0.75, 0.80, 0.85]:
    # Matches only depend on the ratio threshold
    matches = match_descriptors(des1, des2, ratio=ratio)
    
    # The inner loop tests different error tolerances for RANSAC
    for ransac_threshold in [3.0, 5.0, 8.0]:
        pass

```

---

## Handling Failures and Formatting Output

If a ratio is too strict, fewer than 4 matches may survive, causing `estimate_homography` to raise a `ValueError`. Wrapping the estimation step in a `try...except` block ensures the sweep continues across remaining configurations.

```python
print(f"{'ratio':>6} {'ransac':>8} {'matches':>8} {'inliers':>8} {'inlier_ratio':>13}")

for ratio in [0.65, 0.70, 0.75, 0.80, 0.85]:
    matches = match_descriptors(des1, des2, ratio=ratio)
    
    for ransac_threshold in [3.0, 5.0, 8.0]:
        try:
            # Estimate homography and extract inlier mask
            _, inliers = estimate_homography(
                kp1,
                kp2,
                matches,
                ransac_threshold=ransac_threshold,
            )
            
            # Print formatted statistics on success
            print(
                f"{ratio:6.2f} "
                f"{ransac_threshold:8.1f} "
                f"{len(matches):8d} "
                f"{int(inliers.sum()):8d} "
                f"{float(inliers.mean()):13.3f}"
            )
        except ValueError as exc:
            # Catch failures (e.g., < 4 matches) and log failure state
            print(
                f"{ratio:6.2f} "
                f"{ransac_threshold:8.1f} "
                f"{len(matches):8d} "
                f"{'fail':>8} "
                f"{str(exc):>13}"
            )

```

### Key Metrics Computed:

* `inliers.sum()`: Sum of binary mask values ($1$ for inlier, $0$ for outlier) indicating total surviving geometric matches.
* `inliers.mean()`: Ratio of inliers relative to total matches ($\text{inliers} / \text{matches}$).

---

## Example Sweep Output

```text
 ratio   ransac  matches  inliers  inlier_ratio
  0.65      3.0        3     fail  At least four matches are required
  0.70      5.0       45       30         0.667
  0.75      5.0       80       40         0.500

```

---

## Lesson Summary

* **Systematic Evaluation:** Parameter sweeping replaces arbitrary defaults with empirical testing across parameter spaces.
* **Metric Balance:** Prioritize high inlier ratios ($> 50\%$) with adequate inlier volume ($\ge 30$) over raw total matches.
* **Fault Tolerance:** Use `try...except` error boundaries to handle under-constrained configurations gracefully during evaluation loops.

## Sweeping Through Ratio Values

Welcome to the final unit of the course, where the focus shifts from making alignment work once to making it work reliably. Throughout the previous units, you used parameter values like ratio=0.75 without ever asking if that was the best choice for your image pair.

In this exercise, you will take the first step toward building a parameter sweeper: testing several ratio values in a single run instead of just one.

Open solution.py and find the # TODO comment. Currently, the script runs the matching and homography estimation once with a fixed ratio of 0.75.

Your job is to:

    Replace the line ratio = 0.75 with a for loop that iterates through [0.65, 0.70, 0.75, 0.80, 0.85].
    Indent the existing body (the match_descriptors call, the try/except block, and both print calls) so that it runs once per ratio value.

When you run the script, you should see one row of output per ratio, making it easy to spot which value yields the best inlier ratio. This small change turns your script from a one-shot tool into a real diagnostic — a powerful habit for any computer vision pipeline.

```
import argparse

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    print("method:", args.method)
    print(f"{'ratio':>6} {'ransac':>8} {'matches':>8} {'inliers':>8} {'inlier_ratio':>13}")

    ransac_threshold = 5.0

    # TODO: Replace the hardcoded `ratio` line below with a `for` loop that
    # iterates through these values: [0.65, 0.70, 0.75, 0.80, 0.85].
    # Don't forget to indent the body below so it runs once for each value.
    ratio = 0.75
    matches = match_descriptors(des1, des2, ratio=ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=ransac_threshold,
        )
        print(
            f"{ratio:6.2f} "
            f"{ransac_threshold:8.1f} "
            f"{len(matches):8d} "
            f"{int(inliers.sum()):8d} "
            f"{float(inliers.mean()):13.3f}"
        )
    except ValueError as exc:
        print(
            f"{ratio:6.2f} "
            f"{ransac_threshold:8.1f} "
            f"{len(matches):8d} "
            f"{'fail':>8} "
            f"{str(exc):>13}"
        )


if __name__ == "__main__":
    main()

```

Here is the updated `solution.py` with the `ratio` loop implemented:

```python
import argparse

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    print("method:", args.method)
    print(f"{'ratio':>6} {'ransac':>8} {'matches':>8} {'inliers':>8} {'inlier_ratio':>13}")

    ransac_threshold = 5.0

    for ratio in [0.65, 0.70, 0.75, 0.80, 0.85]:
        matches = match_descriptors(des1, des2, ratio=ratio)

        try:
            _, inliers = estimate_homography(
                kp1,
                kp2,
                matches,
                ransac_threshold=ransac_threshold,
            )
            print(
                f"{ratio:6.2f} "
                f"{ransac_threshold:8.1f} "
                f"{len(matches):8d} "
                f"{int(inliers.sum()):8d} "
                f"{float(inliers.mean()):13.3f}"
            )
        except ValueError as exc:
            print(
                f"{ratio:6.2f} "
                f"{ransac_threshold:8.1f} "
                f"{len(matches):8d} "
                f"{'fail':>8} "
                f"{str(exc):>13}"
            )


if __name__ == "__main__":
    main()

```

## Sweeping Through RANSAC Thresholds

Nice work getting the ratio sweep running in the previous exercise — now it is time to make the diagnostic twice as powerful by varying the RANSAC threshold as well.

Currently, ransac_threshold is still a fixed value positioned above the ratio loop. The goal is to turn it into a second axis of the sweep so that every ratio is tested against several threshold values.

Here is what to do:

    Delete the hardcoded ransac_threshold = 5.0 line above the ratio loop.
    Inside the ratio loop (after matches = match_descriptors(...)), add a nested loop: for ransac_threshold in [3.0, 5.0, 8.0]:.
    Indent the existing try/except block one level deeper so that it runs once per threshold value.

Once you are finished, the script will print one row for every (ratio, ransac_threshold) combination — a real parameter sweeper that allows you to spot the sweet spot at a glance.

```
import argparse

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    print("method:", args.method)
    print(f"{'ratio':>6} {'ransac':>8} {'matches':>8} {'inliers':>8} {'inlier_ratio':>13}")

    # TODO: Remove this hardcoded value. The inner `for` loop you add below
    # will supply the `ransac_threshold` for each iteration.
    ransac_threshold = 5.0

    for ratio in [0.65, 0.70, 0.75, 0.80, 0.85]:
        matches = match_descriptors(des1, des2, ratio=ratio)

        # TODO: Add a nested `for` loop that iterates `ransac_threshold`
        # through these values: [3.0, 5.0, 8.0].
        # Then indent the `try/except` block below so it runs once per
        # threshold value (one level deeper than the ratio loop).
        try:
            _, inliers = estimate_homography(
                kp1,
                kp2,
                matches,
                ransac_threshold=ransac_threshold,
            )
            print(
                f"{ratio:6.2f} "
                f"{ransac_threshold:8.1f} "
                f"{len(matches):8d} "
                f"{int(inliers.sum()):8d} "
                f"{float(inliers.mean()):13.3f}"
            )
        except ValueError as exc:
            print(
                f"{ratio:6.2f} "
                f"{ransac_threshold:8.1f} "
                f"{len(matches):8d} "
                f"{'fail':>8} "
                f"{str(exc):>13}"
            )


if __name__ == "__main__":
    main()

```

Here is the updated `solution.py` with the nested loop over `ransac_threshold` values:

```python
import argparse

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    print("method:", args.method)
    print(f"{'ratio':>6} {'ransac':>8} {'matches':>8} {'inliers':>8} {'inlier_ratio':>13}")

    for ratio in [0.65, 0.70, 0.75, 0.80, 0.85]:
        matches = match_descriptors(des1, des2, ratio=ratio)

        for ransac_threshold in [3.0, 5.0, 8.0]:
            try:
                _, inliers = estimate_homography(
                    kp1,
                    kp2,
                    matches,
                    ransac_threshold=ransac_threshold,
                )
                print(
                    f"{ratio:6.2f} "
                    f"{ransac_threshold:8.1f} "
                    f"{len(matches):8d} "
                    f"{int(inliers.sum()):8d} "
                    f"{float(inliers.mean()):13.3f}"
                )
            except ValueError as exc:
                print(
                    f"{ratio:6.2f} "
                    f"{ransac_threshold:8.1f} "
                    f"{len(matches):8d} "
                    f"{'fail':>8} "
                    f"{str(exc):>13}"
                )


if __name__ == "__main__":
    main()

```

## Quiz on Parameter Sweep Basics

**Question 1**

* **Text:** "When building a parameter sweep with nested loops, why is `match_descriptors` called inside the outer loop (ratio loop) rather than inside the inner loop (RANSAC threshold loop)?"
* **Answer:** **Matching only depends on the ratio, not the RANSAC threshold** (Option 3)

---

**Question 2**

* **Text:** "What does a higher ratio value (like 0.85) do when matching descriptors between two images?"
* **Answer:** **It produces more matches but many may be incorrect** (Option 1)

---

**Question 3**

* **Text:** "Why do we wrap the `estimate_homography` call in a `try...except` block during a parameter sweep?"
* **Answer:** **To catch errors when there aren't enough matches for RANSAC** (Option 4)

---

**Question 4**

* **Text:** "When analyzing the results of a parameter sweep, what indicates the best settings for image alignment?"
* **Answer:** **A high inlier ratio alongside a reasonable number of inliers** (Option 1)